# 💻 Notebook 4: Genie Code — Technical Practitioner Demos

## Genie Family PoC Workshop | Manufacturing Industry

**Purpose**: Demonstrate Genie Code capabilities for data engineers, analytics engineers, and data scientists — showing how AI accelerates every technical workflow in Databricks.

**Target personas**: Data Engineers, Analytics Engineers, Data Scientists, ML Engineers

**What this notebook covers**:
1. **Chat Mode** — Code generation, explanation, optimization, and debugging
2. **Agent Mode** — Autonomous multi-step workflows (dashboard creation, pipeline generation)
3. **Dashboard Authoring** — Build AI/BI dashboards with natural language
4. **Genie Agent Authoring** — Create and maintain agents via Genie Code
5. **Custom Instructions & Skills** — Personalize Genie Code behavior
6. **MCP Integration** — Connect to external tools

**Runtime**: ~10 minutes | **Prerequisite**: Run Notebook 1 first

---

### 🎯 What is Genie Code?

Genie Code is the **AI coding and data assistant** for technical practitioners. It:
- Generates and runs code (SQL, Python, Scala)
- Builds pipelines and AI/BI dashboards
- Debugs errors and optimizes queries
- Works with Unity Catalog metadata and lineage
- Operates in **Chat Mode** (interactive) or **Agent Mode** (autonomous)

**Available surfaces**: Notebooks, SQL Editor, Lakeflow Pipelines Editor, AI/BI Dashboards, MLflow, Full-page Command Center

### 💰 Pricing
- **150 free DBUs (~$10) per user per month** — covers ~90% of users
- ~20-30 coding sessions per month within free allowance
- Pay-as-you-go beyond the free tier


## 🗣️ Section 1: Chat Mode Demos

Chat Mode is the interactive experience where you ask Genie Code questions and it generates code, explains concepts, or helps debug issues. These demos are designed to be **run live** — paste the prompts into the Genie Code sidebar.

### How to Open Genie Code
1. In any notebook, click the **Genie Code icon** (✨) in the right sidebar
2. Or navigate to the **full-page command center**: click Genie Code in the left nav
3. Or use keyboard shortcut: `Cmd+I` (Mac) / `Ctrl+I` (Windows)


### Demo 1.1: SQL Generation

**Prompt to paste into Genie Code**:
```
Write a SQL query to find the top 10 plants by production efficiency this month.
Include plant name, region, total production volume, yield rate, quality pass rate,
and equipment uptime. Use the genie_family_poc.manufacturing schema.
```

**What to highlight**:
- Genie Code understands the schema and generates correct joins
- It uses the column comments to understand metric definitions
- The generated SQL is immediately runnable


In [ ]:
# Demo 1.1: The SQL that Genie Code should generate (reference answer)
# Paste the prompt above into Genie Code sidebar and compare

reference_sql = f"""
-- Expected output from Genie Code (may vary slightly)
SELECT
    p.plant_name,
    p.region,
    p.plant_type,
    SUM(o.good_units) AS total_production,
    ROUND(SUM(o.good_units) * 100.0 / NULLIF(SUM(o.planned_quantity), 0), 2) AS yield_rate_pct,
    ROUND(AVG(k.quality_pass_rate_pct), 2) AS avg_quality_pass_rate,
    ROUND(AVG(k.equipment_uptime_pct), 2) AS avg_equipment_uptime
FROM {FULL_SCHEMA}.fct_production_orders o
JOIN {FULL_SCHEMA}.dim_plants p ON o.plant_id = p.plant_id
LEFT JOIN {FULL_SCHEMA}.agg_daily_plant_kpis k
    ON o.plant_id = k.plant_id
    AND o.order_date = k.kpi_date
WHERE o.order_status = 'Completed'
  AND o.order_date >= DATE_TRUNC('month', CURRENT_DATE())
  AND p.is_active = TRUE
GROUP BY p.plant_name, p.region, p.plant_type
ORDER BY yield_rate_pct DESC
LIMIT 10
"""

# Run the reference query to show expected results
display(spark.sql(reference_sql))


### Demo 1.2: Python Code Generation

**Prompt to paste into Genie Code**:
```
Create a Python function that calculates OEE (Overall Equipment Effectiveness)
for each plant using the manufacturing data. OEE = Availability × Performance × Quality.
Use the agg_daily_plant_kpis table. Show results as a formatted DataFrame.
```

**What to highlight**:
- Genie Code generates PySpark code that works with Unity Catalog tables
- It understands the OEE formula from the column comments
- The code is production-quality with error handling


In [ ]:
# Demo 1.2: Reference implementation for OEE calculation
# Genie Code should generate something similar

FULL_SCHEMA = "genie_family_poc.manufacturing"

def calculate_oee(schema=FULL_SCHEMA, days=30):
    """
    Calculate OEE (Overall Equipment Effectiveness) per plant.
    OEE = Availability × Performance × Quality
    Where:
      - Availability = equipment_uptime_pct / 100
      - Performance = yield_rate_pct / 100 (proxy for throughput efficiency)
      - Quality = quality_pass_rate_pct / 100
    """
    df = spark.sql(f"""
        SELECT
            k.plant_id,
            p.plant_name,
            p.region,
            ROUND(AVG(k.equipment_uptime_pct), 2) AS avg_availability_pct,
            ROUND(AVG(k.yield_rate_pct), 2) AS avg_performance_pct,
            ROUND(AVG(k.quality_pass_rate_pct), 2) AS avg_quality_pct,
            ROUND(
                (AVG(k.equipment_uptime_pct) / 100) *
                (AVG(k.yield_rate_pct) / 100) *
                (AVG(k.quality_pass_rate_pct) / 100) * 100
            , 2) AS oee_pct,
            COUNT(*) AS days_measured
        FROM {schema}.agg_daily_plant_kpis k
        JOIN {schema}.dim_plants p ON k.plant_id = p.plant_id
        WHERE k.kpi_date >= CURRENT_DATE() - INTERVAL {days} DAYS
          AND p.is_active = TRUE
        GROUP BY k.plant_id, p.plant_name, p.region
        ORDER BY oee_pct DESC
    """)
    return df

# Run and display
oee_results = calculate_oee()
print("🏭 OEE (Overall Equipment Effectiveness) by Plant — Last 30 Days")
print("   OEE = Availability × Performance × Quality")
print("   Target: >85% (World-class: >90%)")
print()
display(oee_results)


### Demo 1.3: Code Debugging

**Setup**: Run the intentionally broken code below, then ask Genie Code to fix it.

**Prompt to paste into Genie Code**:
```
Fix the error in the cell above. The query has a syntax error and a logic issue.
```

**What to highlight**:
- Genie Code reads the error message and the code
- It identifies both the syntax error AND the logic issue
- It provides a corrected version with explanation


In [ ]:
# Demo 1.3: Intentionally broken code for Genie Code to fix
# This has TWO issues: (1) syntax error (missing comma) (2) wrong join column

FULL_SCHEMA = "genie_family_poc.manufacturing"

try:
    broken_query = f"""
    SELECT
        p.plant_name
        p.region,  -- BUG 1: Missing comma after plant_name
        COUNT(*) AS total_orders,
        SUM(o.good_units) AS total_production
    FROM {FULL_SCHEMA}.fct_production_orders o
    JOIN {FULL_SCHEMA}.dim_products p ON o.plant_id = p.product_id  -- BUG 2: Wrong join
    WHERE o.order_status = 'Completed'
    GROUP BY p.plant_name, p.region
    """
    display(spark.sql(broken_query))
except Exception as e:
    print(f"❌ Error (expected — ask Genie Code to fix this!):")
    print(f"   {str(e)[:200]}")
    print(f"\n💡 Open Genie Code sidebar and ask: 'Fix the error in the cell above'")


## 🤖 Section 2: Agent Mode Demos

Agent Mode is Genie Code's autonomous capability. It can:
- **Plan** a multi-step task
- **Execute** code and inspect results
- **Fix errors** automatically
- **Ask for approval** before consequential actions

### Demo 2.1: Create a Dashboard

**Prompt for Genie Code (Agent Mode)**:
```
Create an AI/BI dashboard called "Manufacturing Operations Overview" with:
1. A KPI row showing total production, yield rate, quality pass rate, and equipment uptime
2. A line chart of daily production volume over the last 90 days
3. A bar chart comparing yield rate by region
4. A table of the bottom 5 plants by yield rate
5. A date range filter

Use data from genie_family_poc.manufacturing schema.
```

**What to highlight**:
- Agent Mode plans all 5 steps before executing
- It creates SQL datasets, builds widgets, and arranges the layout
- It asks for approval before publishing
- The dashboard is immediately usable in Genie One and mobile

### Demo 2.2: Create a Lakeflow Pipeline

**Prompt for Genie Code (Agent Mode)**:
```
Create a Lakeflow DLT pipeline that:
1. Reads from fct_production_orders (bronze)
2. Cleans and validates the data (silver)
3. Aggregates daily KPIs per plant (gold)
4. Materializes the result as a Delta table

Use the genie_family_poc.manufacturing schema.
```

### Demo 2.3: Create a Genie Agent

**Prompt for Genie Code (Agent Mode)**:
```
Create a new Genie Agent called "Quality Control Intelligence" focused on
quality inspection data. Include the quality inspections table, plants table,
and products table from genie_family_poc.manufacturing. Add instructions
about quality metrics and 5 example questions a quality engineer would ask.
```

**What to highlight**:
- Genie Code can create and configure Genie Agents
- It adds appropriate instructions and sample questions
- The agent is immediately available in Genie One


In [ ]:
# Agent Mode reference: Dashboard SQL datasets
# These are the datasets Genie Code would create for Demo 2.1

FULL_SCHEMA = "genie_family_poc.manufacturing"

# Dataset 1: KPI Summary
kpi_sql = f"""
SELECT
    SUM(good_units) AS total_production,
    ROUND(SUM(good_units) * 100.0 / NULLIF(SUM(planned_quantity), 0), 2) AS yield_rate_pct,
    COUNT(DISTINCT plant_id) AS active_plants,
    COUNT(DISTINCT order_id) AS total_orders
FROM {FULL_SCHEMA}.fct_production_orders
WHERE order_status = 'Completed'
  AND order_date >= CURRENT_DATE() - INTERVAL 30 DAYS
"""

# Dataset 2: Daily Production Trend
trend_sql = f"""
SELECT
    order_date,
    SUM(good_units) AS daily_production,
    ROUND(SUM(good_units) * 100.0 / NULLIF(SUM(planned_quantity), 0), 2) AS yield_rate_pct
FROM {FULL_SCHEMA}.fct_production_orders
WHERE order_status = 'Completed'
  AND order_date >= CURRENT_DATE() - INTERVAL 90 DAYS
GROUP BY order_date
ORDER BY order_date
"""

# Dataset 3: Yield by Region
region_sql = f"""
SELECT
    p.region,
    ROUND(SUM(o.good_units) * 100.0 / NULLIF(SUM(o.planned_quantity), 0), 2) AS yield_rate_pct,
    SUM(o.good_units) AS total_production
FROM {FULL_SCHEMA}.fct_production_orders o
JOIN {FULL_SCHEMA}.dim_plants p ON o.plant_id = p.plant_id
WHERE o.order_status = 'Completed'
  AND o.order_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY p.region
ORDER BY yield_rate_pct DESC
"""

print("📊 Dashboard Dataset Previews:")
print("\n1️⃣ KPI Summary:")
display(spark.sql(kpi_sql))
print("\n2️⃣ Daily Production Trend (last 10 days):")
display(spark.sql(trend_sql).orderBy(F.desc("order_date")).limit(10))
print("\n3️⃣ Yield by Region:")
display(spark.sql(region_sql))


## 🛠️ Section 3: Custom Instructions & Skills

### Workspace Instructions
Workspace admins can set instructions that apply to all Genie Code users:

```
File: /Workspace/.genie/instructions.md

# Manufacturing Workspace Instructions

## Data Standards
- Always use the `genie_family_poc.manufacturing` schema for manufacturing data
- Filter to `order_status = 'Completed'` for historical production analysis
- Use `is_active = TRUE` when querying plants (excludes decommissioned facilities)

## Code Standards
- Use PySpark for data transformations, not pandas
- Always include error handling in production code
- Add docstrings to all functions
- Use Delta format for all table writes

## Naming Conventions
- Tables: snake_case (e.g., fct_production_orders)
- Columns: snake_case (e.g., yield_rate_pct)
- Views: prefix with v_ (e.g., v_production_summary)
```

### Personal Skills
Users can create reusable skills that Genie Code invokes automatically:

```
Skill: data-quality-check
Trigger: When user asks to validate or check data quality
Action: Run null checks, duplicate checks, range validation, and freshness checks
```

### MCP Server Integration
Connect Genie Code to external tools:
- **GitHub**: Reference code from repositories
- **Jira**: Create tickets for data quality issues
- **Slack**: Post alerts when pipelines fail


In [ ]:
# Example: Workspace instructions file content
workspace_instructions = """# Manufacturing Workspace Instructions for Genie Code

## Data Access
- Primary schema: `genie_family_poc.manufacturing`
- Always filter production orders to `order_status = 'Completed'` for historical analysis
- Exclude decommissioned plants (`is_active = TRUE`) unless specifically asked

## Metric Definitions
- Yield Rate = good_units / planned_quantity × 100 (target: >95%)
- Scrap Rate = scrap_units / planned_quantity × 100
- Quality Pass Rate = count(result='Pass') / count(*) × 100 (target: >96%)
- Equipment Uptime = count(alert_level='Normal') / count(*) × 100 (target: >90%)
- OEE = Availability × Performance × Quality (target: >85%)

## Code Standards
- Use PySpark for data transformations
- Always include error handling
- Add docstrings to functions
- Use Delta format for table writes
- Follow medallion architecture (bronze → silver → gold)
"""

print("📝 WORKSPACE INSTRUCTIONS")
print("=" * 60)
print(f"Save this content to: /Workspace/.genie/instructions.md")
print()
print(workspace_instructions)


## 🎉 Summary

### What We Demonstrated
- ✅ **Chat Mode**: SQL generation, Python code, debugging
- ✅ **Agent Mode**: Dashboard creation, pipeline generation, agent authoring
- ✅ **Custom Instructions**: Workspace-level coding standards
- ✅ **Reference datasets**: Pre-built SQL for dashboard widgets

### Key Talking Points
1. **Genie Code understands your data**: It reads UC metadata, comments, and lineage
2. **Agent Mode is autonomous**: Plan → Execute → Fix → Approve workflow
3. **One tool, many surfaces**: Notebooks, SQL Editor, Dashboards, Pipelines, MLflow
4. **Governed**: Only accesses data the user has permission for
5. **150 free DBUs/month**: Covers ~90% of users

### Next Steps
- **Notebook 5**: Genie Mobile setup and demo guide
